# 다음 카페 게시판 게시글 제목 크롤링
- 대상 게시판: https://cafe.daum.net/skfootball/IxV3
- 다음 카페는 iframe 구조라 직접 `_c21_` 내부 URL을 사용해야 함

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# 다음 카페 내부 URL 패턴으로 게시판 목록 접근
# grpid=1O7ju (skfootball 카페), fldid=IxV3 (해당 게시판)
url = "https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://cafe.daum.net/skfootball/IxV3"
}

response = requests.get(url, headers=headers)
print("Status:", response.status_code)
print("Content length:", len(response.text))

Status: 200
Content length: 76298


In [3]:
# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

# 페이지 구조 확인
print(soup.prettify()[:3000])

<!DOCTYPE html>
<html class="" lang="ko">
 <head>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="ie=edge" http-equiv="X-UA-Compatible"/>
  <meta content="서경 축구클럽 총모임 - [조기축구동호회친선경기]" property="og:site_name"/>
  <meta content="_서울북부 평일축구" property="og:title"/>
  <meta content="서경 축구클럽 총모임 - [조기축구동호회친선경기]" property="og:article:author"/>
  <meta content="글 작성시 말머리를 달아주세요~!" property="og:description"/>
  <meta content="https://t1.daumcdn.net/cafe_image/cafe_meta_image_240307.png" property="og:image"/>
  <meta content="모든 이야기의 시작, Daum 카페" name="description"/>
  <meta content="noindex,indexifembedded" name="robots"/>
  <meta content="unsafe-url" name="referrer"/>
  <script src="//t1.daumcdn.net/cafe_cj/pcweb/js/1/jquery-1.12.4.min.js">
  </script>
  <script src="//t1.daumcdn.net/cssjs/userAgent/userAgent-1.0.14.min.js" type="text/javascript">
  </script>
  <title>
   서경 축구클럽 총모임 

In [4]:
# 게시글 제목 추출
# 다음 카페 게시판 목록의 제목 태그 선택자
titles = []

# 방법 1: .article-item 클래스
items = soup.select(".article-item")
print(f"article-item 개수: {len(items)}")

for item in items:
    title_tag = item.select_one(".tit-box")
    if title_tag:
        titles.append(title_tag.get_text(strip=True))

print(f"추출된 제목 수: {len(titles)}")
for t in titles[:10]:
    print(t)

article-item 개수: 0
추출된 제목 수: 0


In [5]:
# 방법 2: 다른 선택자 시도 (방법 1이 안 될 경우)
# 링크 태그에서 게시글 관련 것만 추출
all_links = soup.find_all("a")
print(f"전체 링크 수: {len(all_links)}")

# 게시글 링크 필터 (datanum 파라미터 포함)
post_links = [a for a in all_links if a.get("href") and "datanum" in str(a.get("href", ""))]
print(f"게시글 링크 수: {len(post_links)}")

for link in post_links[:10]:
    print(link.get_text(strip=True), "|", link.get("href"))

전체 링크 수: 27
게시글 링크 수: 0


In [6]:
# 방법 3: tbody 내 tr 태그 (테이블 형식 게시판)
rows = soup.select("tbody tr")
print(f"tr 행 수: {len(rows)}")

for row in rows[:10]:
    print(row.get_text(strip=True)[:100])

tr 행 수: 0


## 디버깅: 실제로 뭐가 HTML에 들어있나 확인

In [7]:
# 27개 링크 전체 출력
for a in soup.find_all("a"):
    print(repr(a.get_text(strip=True)[:50]), "|", a.get("href", "")[:100])

'다음 카페의 ie10 이하 브라우저 지원이 종료됩니다. 원활한 카페 이용을 위해 사용 중인' | https://cafe.daum.net/supporters/MbmU/150
'Daum' | https://www.daum.net/?t__nil_navi=daum
'카페' | https://top.cafe.daum.net?t__nil_navi=cafehome
'테이블' | https://table.cafe.daum.net
'메일' | https://mail.daum.net?t__nil_navi=mailhome
'즐겨찾는 카페' | #
'로그인' | #
'최신글 보기' | /_c21_/recent_bbs_list?grpid=1O7ju&fldid=_rec
'인기글 보기' | /_c21_/favor_bbs_list?grpid=1O7ju
'공지사항' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUh
'우리팀 소개' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUg
'용병 신청&구함' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUe
'플래티넘 (공개)' | /_c21_/cafe_profile?grpid=1O7ju
'공차야산다' | #
'회원수' | /_c21_/memberlist?grpid=1O7ju
'84,581' | /_c21_/memberlist?grpid=1O7ju
'카페앱수' | https://cafe-notice.tistory.com/2328
'' | /_c21_/join_register?grpid=1O7ju
'검색' | #
'카페 전체 메뉴' | #
'▲' | #
'서비스 약관/정책' | https://policy.daum.net/policy/info
'권리침해신고' | https://cs.daum.net/redbell/top.html
'이용약관' | https://top.cafe.daum.net/_c21_/agreement_axz
'카페 고객센터' | https://cs.daum.net/f

In [8]:
import re

raw = response.text

# raw HTML에 datanum 있는지 확인
print("datanum in raw HTML:", "datanum" in raw)

datanums = re.findall(r'datanum[=:]+(\d+)', raw)
print("발견된 datanum들:", datanums[:20])

# bbs_read 링크 패턴
bbs_reads = re.findall(r'bbs_read[^"\'\\s]*', raw)
print("\nbbs_read 패턴들:", bbs_reads[:10])

datanum in raw HTML: False
발견된 datanum들: []

bbs_read 패턴들: ['bbs_read a.btn:fir']


## bbs_read 방식으로 전환
crawling_trial1에서 확인: `_c21_/bbs_read` URL은 requests만으로 실제 HTML 반환됨  
→ 게시글 상세 페이지 하단에 같은 게시판 글 목록이 포함되어 있어서 datanum 목록 수집 가능

In [9]:
# bbs_list 에서 datanum을 못 찾으면 bbs_read로 IxV3 게시판 아무 글이나 찾기
# bbs_list raw HTML에서 IxV3 관련 datanum 추출 시도
if datanums:
    start_datanum = datanums[0]
    print(f"bbs_list에서 찾은 datanum: {start_datanum}")
else:
    # bbs_list 페이지의 og:title 이 "_서울북부 평일축구" 처럼 게시글 제목이 있으므로
    # 해당 게시글이 최근 글임 → datanum을 모르니 bbs_list HTML 더 탐색
    print("datanum 없음, HTML 끝부분 확인")
    print(raw[-3000:])

datanum 없음, HTML 끝부분 확인
\uB9AC\uC218 \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'TJDk',
			fldname: '\u25A0\u25A0 \uC2DC\/\uB3C4\/\uAD6C\/\uAD70 \uD611\uD68C\uB300\uD68C(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'U9qk',
			fldname: '\u25A0\u25A0 \uC720\uC18C\uB144 \uB300\uD68C&\uAD50\uB958\uC804(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'UPDv',
			fldname: '\u25A0\u25A0 2016 \uC81C2\uD68C \uAC8C\uD1A0\uB808\uC774 GUFA CUP(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'Sp61',
			fldname: '\u25A1\u25A1 2015 \uC11C\uC6B8 \uC2A4\uB9C8\uC77C \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'TNqN',
			fldname: '\u25A1\u25A1 2014 \uB4DC\uB9BC\uCEF5 \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'Uc84',
			fldname: '_VENTUS',
		})
							boards.push({
			fldid: 'UcNt',
			fldname: '_\uBCA4\uD22C\uC2A4SC',
		})
			</script>

<link rel="stylesh

In [10]:
# start_datanum을 위 셀 결과로 채우거나, 직접 지정
# (브라우저로 https://cafe.daum.net/skfootball/IxV3 열어서 아무 글 URL의 숫자 확인)
start_datanum = 136431  # 또는 직접 입력: "12345"

test_url = f"https://cafe.daum.net/_c21_/bbs_read?grpid=1O7ju&fldid=IxV3&datanum={start_datanum}"
print("접속 URL:", test_url)

res2 = requests.get(test_url, headers=headers)
print("Status:", res2.status_code, "| 크기:", len(res2.text))

soup2 = BeautifulSoup(res2.text, "html.parser")

og_title = soup2.find("meta", {"property": "og:title"})
print("og:title:", og_title["content"] if og_title else "없음")

접속 URL: https://cafe.daum.net/_c21_/bbs_read?grpid=1O7ju&fldid=IxV3&datanum=136431
Status: 404 | 크기: 4657
og:title: 없음


In [11]:
# bbs_read 페이지에서 같은 게시판의 글 목록 datanum 전체 수집
raw2 = res2.text

all_datanums_in_page = list(set(re.findall(r'datanum[=:]+(\d+)', raw2)))
print(f"페이지 내 datanum {len(all_datanums_in_page)}개:", all_datanums_in_page[:20])

# bbs_read 링크 패턴으로 글 목록 링크 추출
bbs_read_in_page = re.findall(r'bbs_read\?[^"\'<\s]+', raw2)
print(f"\nbbs_read 링크 {len(bbs_read_in_page)}개:")
for link in bbs_read_in_page[:20]:
    print(" ", link)

페이지 내 datanum 0개: []

bbs_read 링크 0개:


## HTML에 articles JS 변수가 있는지 확인
bbs_list 페이지 하단 config에 `articles: articles`가 있음 → JS 변수로 미리 주입돼 있을 가능성

In [12]:
# articles.push 패턴 탐색
articles_pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
print(f"articles.push 개수: {len(articles_pushes)}")
for a in articles_pushes[:3]:
    print(a[:300])
    print()

# var articles 선언 탐색
idx = raw.find("var articles")
if idx != -1:
    print("var articles 발견:")
    print(raw[idx:idx+500])
else:
    print("var articles 없음")

articles.push 개수: 20
articles.push({
		dataid: '24681',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Q5zzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '6\uC6D4 11\uC77C, 25\uC77C \uBAA9\uC694\uC77C 21\uC2DC~23\uC2DC \uC911\uB791\uAD6C\uB9BD\uC794\uB514\uC6B4\uB3D9\uC7A5',
		headCont: '\u25B6\uCD08\uCCAD\uD569\uB2C8\uB2E4',


articles.push({
		dataid: '24680',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Q4zzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '5\/22 \uAE08\uC694\uC77C \uACBD\uAE30 \uCD08\uCCAD\uC694\uCCAD',
		headCont: '',
		board: '',
		vldstatus: '',
		commentCnt: 0,
		isNewComment: '' == 'Y',

		created: '26.05

articles.push({
		dataid: '24679',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Q3zzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '6\/5 \uAE08\uC694\uC77C , 6\/9 \uD654\uC694\uC77C 20-22 \uCD08\uCCAD\uD569\uB2C8\uB2E4 (\uAD50\uD658\uACBD\uAE30\uC6B0\uC120)',
		headCont: '',
		board: '',
		vldstatus: '',

var articles 발견:
var articles = [];
		articles.push({
		dataid: '24681',


In [25]:
import json
from datetime import datetime

titles_from_js = []
for item in articles_pushes:
    dataid_m  = re.search(r"dataid:\s*'(\d+)'", item)
    title_m   = re.search(r"title:\s*'([^']*)'", item)
    created_m = re.search(r"created:\s*'([^']+)'", item)
    if dataid_m and title_m:
        raw_title = title_m.group(1)
        decoded_title = json.loads(f'"{raw_title}"')
        created_str = created_m.group(1) if created_m else None
        created_dt  = datetime.strptime(created_str, '%y.%m.%d') if created_str else None
        titles_from_js.append({
            "dataid": dataid_m.group(1),
            "title": decoded_title,
            "created": created_dt,
        })

print(f"추출된 제목 {len(titles_from_js)}개:")
for t in titles_from_js:
    print(t["dataid"], "|", t["created"].strftime("%Y-%m-%d") if t["created"] else "-", "|", t["title"])

추출된 제목 20개:
24681 | 2026-05-21 | 6월 11일, 25일 목요일 21시~23시 중랑구립잔디운동장
24680 | 2026-05-21 | 5/22 금요일 경기 초청요청
24679 | 2026-05-21 | 6/5 금요일 , 6/9 화요일 20-22 초청합니다 (교환경기우선)
24678 | 2026-05-21 | 노원구 6월 18일 (목) 20시 - 22시 불암산 구장 교환경기 모십니다
24677 | 2026-05-21 | 6월4일 목요일 오후8시 효창구장 (마감)
24676 | 2026-05-21 | 6월 평일 야간 초청 합니다
24675 | 2026-05-21 | 6월5일 금 21시~23시 중랑구립잔디운동장
24674 | 2026-05-21 | 6월 22일 월요일 우장산축구장 20~22시 초청합니다.
24673 | 2026-05-21 | 2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.
24672 | 2026-05-20 | (강서구) 6/30 (화) 가양구장 20-22시 초청합니다. 완료
24670 | 2026-05-20 | 2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.
24669 | 2026-05-20 | [마감/마감]5월 21일 목요일 21~23시 용마폭포공원축구장 상대 팀 모십니다.
24668 | 2026-05-20 | [노원구] 5월 27일(수) 20시 초청합니다.
24667 | 2026-05-20 | 05월 25일 월요일 21시~23시 용마폭포공원축구장 초청합니다.
24665 | 2026-05-20 | 5.25 월 오후 5-7 제2월곡 초청
24663 | 2026-05-20 | 26.05.25(월, 대체공휴일) 새벽 06-08시 의정부 경민대학교 인조잔디구장 11:11 무료초청
24662 | 2026-05-19 | 5월26일(화) 21시 남양주크낙새구장(2월잔디교체) 무료초청합니다.
24661 | 2026-05-19 | 5월22일(금)/21-23시/국민대학교 

## 전체 페이지 수집 (1페이지 20개 × 3페이지)
`page=N` 파라미터로 다음 페이지 접근 가능

In [38]:
import time
from datetime import datetime

def _parse_pushes(raw):
    """articles.push(...) 블록에서 dataid / title / created 추출"""
    pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
    result = []
    for item in pushes:
        dm = re.search(r"dataid:\s*'(\d+)'", item)
        tm = re.search(r"title:\s*'([^']*)'", item)
        cm = re.search(r"created:\s*'([^']+)'", item)
        if dm and tm:
            created_str = cm.group(1) if cm else None
            created_dt  = datetime.strptime(created_str, '%y.%m.%d') if created_str else None
            result.append({
                "dataid": dm.group(1),
                "title": json.loads(f'"{tm.group(1)}"'),
                "created": created_dt,
            })
    return result


def _get_page(page_num, prev_page, first_depth, last_depth):
    url = (
        f"https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3"
        f"&page={page_num}&prev_page={prev_page}&listnum=20"
        f"&firstbbsdepth={first_depth}&lastbbsdepth={last_depth}"
    )
    raw = requests.get(url, headers=headers).text
    new_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw)
    new_last  = re.search(r"lastBbsDepth:\s*'([^']+)'", raw)
    return (
        _parse_pushes(raw),
        page_num,
        new_first.group(1) if new_first else None,
        new_last.group(1)  if new_last  else None,
    )


def collect_posts(max_pages=4, date_from=None, date_to=None):
    """
    게시글 제목 + 작성일 수집.

    Parameters
    ----------
    max_pages : 수집할 최대 페이지 수 (페이지당 20개)
    date_from : 시작일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)
    date_to   : 종료일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)

    Returns
    -------
    list of dict  {"dataid", "title", "created"}
    """
    from_dt = datetime.strptime(date_from, '%Y-%m-%d') if date_from else None
    to_dt   = datetime.strptime(date_to,   '%Y-%m-%d') if date_to   else None

    # 1페이지
    raw0 = requests.get(
        "https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3",
        headers=headers
    ).text
    all_posts = _parse_pushes(raw0)
    cur_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw0).group(1)
    cur_last  = re.search(r"lastBbsDepth:\s*'([^']+)'",  raw0).group(1)
    cur_page  = 1
    print(f"page 1 — {len(all_posts)}개 수집")

    # 2페이지~
    for _ in range(max_pages - 1):
        time.sleep(1)
        posts, cur_page, cur_first, cur_last = _get_page(
            cur_page + 1, cur_page, cur_first, cur_last
        )
        print(f"page {cur_page} — {len(posts)}개 수집")
        all_posts.extend(posts)
        if not cur_first:
            print("마지막 페이지 도달")
            break

    print(f"\n총 수집: {len(all_posts)}개")

    # 날짜 필터
    if from_dt or to_dt:
        filtered = [
            p for p in all_posts
            if p["created"] is not None
            and (from_dt is None or p["created"] >= from_dt)
            and (to_dt   is None or p["created"] <= to_dt)
        ]
        print(f"날짜 필터 적용 ({date_from} ~ {date_to}): {len(filtered)}개")
        return filtered

    return all_posts


# ── 실행: 원하는 기간만 바꿔서 사용 ──────────────────────────────
# 전체 수집:      collect_posts(max_pages=4)
# 기간 지정:      collect_posts(max_pages=10, date_from="2026-05-01", date_to="2026-05-22")
all_posts = collect_posts(max_pages=4, date_from="2026-05-01", date_to="2026-05-22")


page 1 — 20개 수집
page 2 — 20개 수집
page 3 — 20개 수집
page 4 — 20개 수집

총 수집: 80개
날짜 필터 적용 (2026-05-01 ~ 2026-05-22): 80개


## 1페이지 제목 파싱 — 날짜 / 지역 / 시간 / 초청 구분

In [39]:
def parse_title(title):
    result = {"title": title, "날짜": None, "지역": None, "장소": None, "시간": None, "초청구분": None}

    date_patterns = [
        r'\d{2}\.\d{2}\.\d{2}',
        r'\d{1,2}월\s*\d{1,2}일',
        r'\d{1,2}\.\d{1,2}\b',
        r'\d{1,2}\/\d{1,2}',
    ]
    for pat in date_patterns:
        m = re.search(pat, title)
        if m:
            result["날짜"] = m.group(0)
            break

    time_patterns = [
        r'(?:오전|오후|새벽|저녁)?\s*\d{1,2}:\d{2}\s*[~\-]\s*\d{1,2}:\d{2}',
        r'(?:오전|오후|새벽|저녁)?\s*\d{1,2}\s*[시~\-]\s*\d{1,2}\s*시?',
        r'(?:오전|오후|새벽|저녁)\s*\d{1,2}',
    ]
    for pat in time_patterns:
        m = re.search(pat, title)
        if m:
            result["시간"] = m.group(0).strip()
            break

    bracket_m = re.search(r'[\[\(]([가-힣]{2,6}[구시군])[\]\)]', title)
    if bracket_m:
        result["지역"] = bracket_m.group(1)
    else:
        area_m = re.search(r'([가-힣]{2,5}[구시군])[\s\W]', title)
        if area_m:
            result["지역"] = area_m.group(1)

    venue_m = re.search(
        r'[가-힣a-zA-Z0-9]+\s*(?:구장|운동장|체육공원|체육관|인조잔디구장|잔디구장|대운동장|에코랜드|공원)',
        title
    )
    if venue_m:
        result["장소"] = venue_m.group(0).strip()

    if re.search(r'초청\s*(요청|해주|구함|구합)', title):
        result["초청구분"] = "요청"
    elif re.search(r'모십니다|구합니다|구해요|찾습니다|매칭\s*구', title):
        result["초청구분"] = "요청"
    elif re.search(r'초청|무료초청', title):
        result["초청구분"] = "초청"
    else:
        result["초청구분"] = "기타"

    return result


parsed = [parse_title(p["title"]) for p in all_posts]
df = pd.DataFrame(parsed)
df.insert(0, "dataid",  [p["dataid"]  for p in all_posts])
df.insert(1, "created", [p["created"] for p in all_posts])

print(f"총 {len(df)}개")
df


총 80개


,dataid,created,title,날짜,지역,장소,시간,초청구분
0,24681,2026-05-21,"6월 11일, 25일 목요일 21시~23시 중랑구립잔디운동장",6월 11일,NaN,중랑구립잔디운동장,NaN,기타
1,24680,2026-05-21,5/22 금요일 경기 초청요청,5/22,NaN,NaN,NaN,요청
2,24679,2026-05-21,"6/5 금요일 , 6/9 화요일 20-22 초청합니다 (교환경기우선)",6/5,NaN,NaN,20-22,초청
3,24678,2026-05-21,노원구 6월 18일 (목) 20시 - 22시 불암산 구장 교환경기 모십니다,6월 18일,노원구,불암산 구장,NaN,요청
4,24677,2026-05-21,6월4일 목요일 오후8시 효창구장 (마감),6월4일,NaN,효창구장,오후8,기타
...,...,...,...,...,...,...,...,...
75,24583,2026-05-05,5월8일 금요일 20-22 초청합니다,5월8일,NaN,NaN,20-22,초청
76,24582,2026-05-05,5/13 수요일 저녁8-10시 우장산 초청 - 마감,5/13,NaN,NaN,저녁8-10시,초청
77,24581,2026-05-05,마감 5.25 월 중랑구립 15-17시 초청합니다.,5.25,NaN,NaN,15-17시,초청
78,24577,2026-05-04,5월 28일 목요일 20~22 불암산종합스타디움 초청합니다.,5월 28일,NaN,NaN,20~22,초청


In [40]:
# ── 구장명 → 시/군/구 정적 매핑 ────────────────────────────────────
# 키워드(공백 제거 후 부분일치)로 검색. 장소 컬럼이 NaN이거나 너무 generic하면 title에서 fallback.
VENUE_DISTRICT = {
    # 서울 노원구
    "초안산":     ("서울", "노원구"),
    "불암산":     ("서울", "노원구"),
    # 서울 중랑구
    "중랑구립":   ("서울", "중랑구"),
    "용마폭포":   ("서울", "중랑구"),
    # 서울 강북구
    "강북구민":   ("서울", "강북구"),
    # 서울 성북구
    "국민대학교": ("서울", "성북구"),
    "월곡":       ("서울", "성북구"),
    # 서울 용산구
    "효창":       ("서울", "용산구"),
    # 서울 강서구
    "우장산":     ("서울", "강서구"),
    "가양":       ("서울", "강서구"),
    "개화":       ("서울", "강서구"),
    # 서울 성동구
    "응봉":       ("서울", "성동구"),
    # 서울 은평구
    "은평":       ("서울", "은평구"),
    # 서울 마포구
    "월드컵":     ("서울", "마포구"),
    "상암":       ("서울", "마포구"),
    # 서울 광진구
    "아차산":     ("서울", "광진구"),
    # 경기 의정부시
    "경민대학교": ("경기", "의정부시"),
    # 경기 남양주시
    "크낙새":     ("경기", "남양주시"),
    "별내":       ("경기", "남양주시"),
    "다산":       ("경기", "남양주시"),
    # 경기 구리시
    "구리타워":   ("경기", "구리시"),
    "구리":       ("경기", "구리시"),
}

# 너무 generic해서 단독으로 의미없는 장소명
_GENERIC_VENUES = {"축구장", "운동장", "구장", "체육공원", "공원"}

def lookup_district(venue, title=""):
    """
    장소명 → (시도, 시군구).
    1차: venue 컬럼에서 키워드 검색
    2차: venue가 NaN이거나 generic하면 title에서 fallback 검색
    """
    candidates = []

    # 1차: venue 컬럼
    if not pd.isna(venue) and venue.strip() not in _GENERIC_VENUES:
        candidates.append(venue)

    # 2차: title fallback
    if title:
        candidates.append(title)

    for text in candidates:
        normalized = text.replace(" ", "")
        for keyword, (sido, sigungu) in VENUE_DISTRICT.items():
            if keyword in normalized:
                return sido, sigungu

    return None, None


result = df.apply(lambda row: lookup_district(row["장소"], row["title"]), axis=1)
df["시도"], df["시군구"] = zip(*result)

mapped   = df["시군구"].notna().sum()
unmapped = df["시군구"].isna().sum()
print(f"매핑 성공: {mapped}개 / 실패(NaN): {unmapped}개")

if unmapped:
    print("\n[매핑 실패 — 추가 딕셔너리 필요 목록]")
    print(df.loc[df["시군구"].isna(), ["dataid", "장소", "title"]].to_string(index=False))

df[["dataid", "created", "title", "장소", "시도", "시군구"]].head(20)


매핑 성공: 55개 / 실패(NaN): 25개

[매핑 실패 — 추가 딕셔너리 필요 목록]
dataid   장소                                      title
 24680  NaN                           5/22 금요일 경기 초청요청
 24679  NaN     6/5 금요일 , 6/9 화요일 20-22 초청합니다 (교환경기우선)
 24676  NaN                            6월 평일 야간 초청 합니다
 24668  NaN                 [노원구] 5월 27일(수) 20시 초청합니다.
 24659  NaN                      6월 평일 저녁20-22시 초청합니다.
 24657  NaN                5월 21일 목요일 20-22 노원구 초청합니다.
 24656  NaN    6월 평일 월요일 저녁 8시 노원구 축구 초청 및 초청 요청 드립니다.
 24646 문원구장          5/25(월/대체공휴일) 오전06~08시 과천 문원구장 초청
 24639  NaN           5/21(목) 20-22시 노원 마들 스타디움 매치팀 초청
 24638  NaN               5월 28일 목요일 20-22시 노원구 초청합니다!
 24631  NaN                 [노원구] 5월 20일(수) 20시 초청합니다.
 24630  NaN 2026년 5월 노원구 평일, 주말 20:00~22:00 경기 초청드립니다.
 24622  NaN                                   6월 초청합니다
 24620  NaN                                         마감
 24608  NaN                                       매칭완료
 24606  NaN 2026년 5월 노원구 평일, 주말 20:00~22:00 경기 초청드립니다.
 24605  NaN   

,dataid,created,title,장소,시도,시군구
0,24681,2026-05-21,"6월 11일, 25일 목요일 21시~23시 중랑구립잔디운동장",중랑구립잔디운동장,서울,중랑구
1,24680,2026-05-21,5/22 금요일 경기 초청요청,NaN,NaN,NaN
2,24679,2026-05-21,"6/5 금요일 , 6/9 화요일 20-22 초청합니다 (교환경기우선)",NaN,NaN,NaN
3,24678,2026-05-21,노원구 6월 18일 (목) 20시 - 22시 불암산 구장 교환경기 모십니다,불암산 구장,서울,노원구
4,24677,2026-05-21,6월4일 목요일 오후8시 효창구장 (마감),효창구장,서울,용산구
5,24676,2026-05-21,6월 평일 야간 초청 합니다,NaN,NaN,NaN
6,24675,2026-05-21,6월5일 금 21시~23시 중랑구립잔디운동장,중랑구립잔디운동장,서울,중랑구
7,24674,2026-05-21,6월 22일 월요일 우장산축구장 20~22시 초청합니다.,우장산축구장,서울,강서구
8,24673,2026-05-21,2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.,초안산 구장,서울,노원구
9,24672,2026-05-20,(강서구) 6/30 (화) 가양구장 20-22시 초청합니다. 완료,가양구장,서울,강서구


In [41]:
venue_count = (
    df["시군구"]
    .value_counts(dropna=False)
    .rename_axis("장소")
    .reset_index(name="게시물수")
)
venue_count

,장소,게시물수
0,NaN,25
1,노원구,19
2,중랑구,9
3,강서구,6
4,강북구,6
5,성북구,4
6,남양주시,3
7,광진구,2
8,용산구,1
9,의정부시,1


In [43]:
df[["dataid", "created", "title", "장소", "시도", "시군구"]].to_clipboard()